In [1]:
import os
import cv2
import numpy as np
import matplotlib.pyplot as plt
from tqdm import tqdm
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader


In [4]:
class DenseBlock(nn.Module):
    def __init__(self, in_channels, growth_rate=32):
        super(DenseBlock, self).__init__()
        self.conv1 = nn.Conv2d(in_channels, growth_rate, 3, padding=1)
        self.conv2 = nn.Conv2d(in_channels + growth_rate, growth_rate, 3, padding=1)
        self.relu = nn.ReLU(inplace=True)
        self.out_channels = in_channels + 2 * growth_rate

    def forward(self, x):
        x1 = self.relu(self.conv1(x))
        x2 = self.relu(self.conv2(torch.cat([x, x1], dim=1)))
        return torch.cat([x, x1, x2], dim=1)

class SDSC_UNet(nn.Module):
    def __init__(self):
        super(SDSC_UNet, self).__init__()
        # Encoder
        self.enc1 = DenseBlock(3)         # 67 out
        self.pool = nn.MaxPool2d(2)
        self.enc2 = DenseBlock(67)        # 131 out
        self.enc3 = DenseBlock(131)       # 195 out
        self.bottleneck = DenseBlock(195) # 259 out

        # Decoder
        self.up3 = nn.ConvTranspose2d(259, 195, 2, stride=2)
        self.dec3 = nn.Conv2d(390, 195, 3, padding=1) # 195 + 195 skip
        self.up2 = nn.ConvTranspose2d(195, 131, 2, stride=2)
        self.dec2 = nn.Conv2d(262, 131, 3, padding=1) # 131 + 131 skip
        self.up1 = nn.ConvTranspose2d(131, 67, 2, stride=2)
        self.dec1 = nn.Conv2d(134, 64, 3, padding=1)  # 67 + 67 skip

        self.final = nn.Conv2d(64, 1, 1)
        self.relu = nn.ReLU(inplace=True)

    def forward(self, x):
        e1 = self.enc1(x)
        e2 = self.enc2(self.pool(e1))
        e3 = self.enc3(self.pool(e2))
        b = self.bottleneck(self.pool(e3))

        d3 = self.relu(self.dec3(torch.cat([self.up3(b), e3], dim=1)))
        d2 = self.relu(self.dec2(torch.cat([self.up2(d3), e2], dim=1)))
        d1 = self.relu(self.dec1(torch.cat([self.up1(d2), e1], dim=1)))
        return self.final(d1)

In [5]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model = SDSC_UNet()   # your architecture
model.load_state_dict(torch.load("/content/drive/MyDrive/sdsc_unet__whu.pth", map_location=device))
model.to(device)
model.eval()

SDSC_UNet(
  (enc1): DenseBlock(
    (conv1): Conv2d(3, 32, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
    (conv2): Conv2d(35, 32, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
    (relu): ReLU(inplace=True)
  )
  (pool): MaxPool2d(kernel_size=2, stride=2, padding=0, dilation=1, ceil_mode=False)
  (enc2): DenseBlock(
    (conv1): Conv2d(67, 32, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
    (conv2): Conv2d(99, 32, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
    (relu): ReLU(inplace=True)
  )
  (enc3): DenseBlock(
    (conv1): Conv2d(131, 32, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
    (conv2): Conv2d(163, 32, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
    (relu): ReLU(inplace=True)
  )
  (bottleneck): DenseBlock(
    (conv1): Conv2d(195, 32, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
    (conv2): Conv2d(227, 32, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
    (relu): ReLU(inplace=True)
  )
  (up3): ConvTranspose2d(259, 195, 

In [6]:
class WHUDataset(Dataset):
    def __init__(self, image_dir, mask_dir):
        self.image_dir = image_dir
        self.mask_dir = mask_dir
        self.images = sorted(os.listdir(image_dir))

    def __len__(self):
        return len(self.images)

    def __getitem__(self, index):
        img_path = os.path.join(self.image_dir, self.images[index])
        mask_path = os.path.join(self.mask_dir, self.images[index])

        # Load and preprocess image
        image = cv2.imread(img_path)
        image = cv2.cvtColor(image, cv2.COLOR_BGR2RGB)
        image = cv2.resize(image, (256, 256))
        image = image / 255.0

        # Load and preprocess mask
        mask = cv2.imread(mask_path, 0)
        mask = cv2.resize(mask, (256, 256))
        mask = mask / 255.0
        mask = np.expand_dims(mask, axis=0)

        image = torch.tensor(image, dtype=torch.float32).permute(2,0,1)
        mask = torch.tensor(mask, dtype=torch.float32)

        return image, mask



In [7]:
from torch.utils.data import DataLoader

test_dataset = WHUDataset(
    image_dir="/content/drive/MyDrive/WHU/WHU/test/Image",
    mask_dir="/content/drive/MyDrive/WHU/WHU/test/Mask"
)


In [8]:
test_loader = DataLoader(
    test_dataset,
    batch_size=4,
    shuffle=False,
    num_workers=2,
    pin_memory=True
)

In [11]:
import torch
from tqdm import tqdm
import time

def evaluate_model(model, test_loader, device, criterion=None):

    model.eval()

    TP = FP = FN = TN = 0
    total_loss = 0

    start_time = time.time()

    with torch.no_grad():
        for batch_idx, (images, masks) in enumerate(tqdm(test_loader, desc="Testing Progress", leave=True)):

            images = images.to(device, non_blocking=True)
            masks = masks.to(device, non_blocking=True)

            outputs = model(images)

            # Optional loss calculation
            if criterion is not None:
                loss = criterion(outputs, masks)
                total_loss += loss.item()

            # Binary segmentation
            preds = torch.sigmoid(outputs)
            preds = (preds > 0.5).float()

            TP += (preds * masks).sum().item()
            FP += (preds * (1 - masks)).sum().item()
            FN += ((1 - preds) * masks).sum().item()
            TN += ((1 - preds) * (1 - masks)).sum().item()

    # ---- Metrics ----
    accuracy  = (TP + TN) / (TP + TN + FP + FN + 1e-8)
    precision = TP / (TP + FP + 1e-8)
    recall    = TP / (TP + FN + 1e-8)
    f1        = 2 * precision * recall / (precision + recall + 1e-8)
    iou       = TP / (TP + FP + FN + 1e-8)
    dice      = 2 * TP / (2 * TP + FP + FN + 1e-8)

    avg_loss = total_loss / len(test_loader) if criterion is not None else 0
    total_time = time.time() - start_time

    # ---- Print Results ----
    print("\n" + "="*40)
    print("📊 FINAL MODEL PERFORMANCE")
    print("="*40)

    if criterion is not None:
        print(f"Loss      : {avg_loss:.4f}")

    print(f"Accuracy  : {accuracy:.4f}")
    print(f"Precision : {precision:.4f}")
    print(f"Recall    : {recall:.4f}")
    print(f"F1 Score  : {f1:.4f}")
    print(f"IoU       : {iou:.4f}")
    print(f"Dice      : {dice:.4f}")

    print("="*40)
    print(f"⏱ Total Testing Time: {total_time:.2f} seconds")
    print("="*40)

    return accuracy, precision, recall, f1, iou, dice

In [13]:
evaluate_model(model, test_loader, device)

Testing Progress: 100%|██████████| 307/307 [00:26<00:00, 11.45it/s]


📊 FINAL MODEL PERFORMANCE
Accuracy  : 0.9744
Precision : 0.9399
Recall    : 0.8920
F1 Score  : 0.9153
IoU       : 0.8438
Dice      : 0.9153
⏱ Total Testing Time: 26.82 seconds


(0.9743770057323634,
 0.9398937260511238,
 0.8919667952887389,
 0.9153033000002706,
 0.8438334045022503,
 0.9153033049968484)

In [ ]:
!nvidia-smi